# TN1 — Kiến trúc: TCN so với DS-TCN so với LSTM

Đổi **đúng một biến** so với TN0: thay LSTM của MobiVital bằng TCN. Mọi thứ khác giữ nguyên cấu hình tác giả công bố.

## Câu hỏi

**Thu nhỏ model 90–96% thì còn dự báo tốt hơn LSTM không?**

| model | kênh | tham số | so LSTM |
|---|---|---|---|
| LSTM (MobiVital) | hidden 352 | 1.502.713 | — |
| TCN | 64 | 151.513 | −90% |
| DS-TCN | 64 | 56.281 | −96% |

## Ba notebook, chạy theo thứ tự này

| | notebook | chạy gì | thời gian |
|---|---|---|---|
| 1 | `TN1_LSTM.ipynb` | LSTM: 4 fold CV, rồi 3 seed test GHIJ | ~3 h |
| 1 | `TN1_TCN_DSTCN_model_selection.ipynb` | TCN-64 và DS-TCN-64: 4 fold CV | ~2.2 h |
| 2 | **`TN1_final_evaluation.ipynb`** ← đang mở | gộp kết quả, so sánh, GHIJ cho kiến trúc thắng | ~1.5 h |

Hai notebook đầu chạy **song song** ở hai phiên Colab khác nhau, không cần chờ nhau. Mỗi cái nén kết quả ra một tệp zip riêng trên Drive:

```
tn1_lstm.zip    runs/tn1/ (phần lstm) · runs/tn1_ghij/ · summary.csv
tn1_tcn.zip     runs/tn1/ (phần tcn và ds_tcn) · summary.csv
```

Notebook này bung cả hai rồi gộp lại.

## Kết quả hai notebook kia đã cho

| cấu hình | tham số | cv_score |
|---|---|---|
| LSTM | 1.502.713 | **0.760698** |
| DS-TCN-64 | 56.281 | 0.741376 |
| TCN-64 | 151.513 | 0.737742 |

Cả hai TCN đều thấp hơn LSTM, nhưng khoảng thua (0.019 và 0.023) **nhỏ hơn nhiễu hạt giống đo được ở phần LSTM là 0.028**. Nên chưa kết luận được, và đó là lý do mục 4 chạy thêm ba seed.

## Giao thức

Bốn fold cố định trên `A B C D E F K L`. **`G H I J` chỉ đụng ở mục 4**, và chỉ để làm mốc kiểm chứng — quyết định chọn kiến trúc phải dựa trên `cv_score`, xem `docs/PROTOCOL.md`.


## 1. Chuẩn bị Colab


In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
# Xoá trước để chạy lại ô này luôn lấy mã mới nhất, không dính bản cũ.
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py


/content/UWB_RADAR

thư mục làm việc : /content/UWB_RADAR
commit đồ án     : 1ba4e8a
commit MobiVital : 4319731 (đã ghim)
GPU              : NVIDIA L4, 23034 MiB


In [3]:
!python scripts/restore_processed_data_on_drive.py


by_user   : bung /content/drive/MyDrive/mobivital/by_user.tar ...
            12 tệp
windows   : bung /content/drive/MyDrive/mobivital/windows.tar.gz ...
            dev_cv 8 tệp, final_train có

2.5G	data/processed/by_user
503M	data/processed/windows


## 2. Gộp kết quả hai phiên

Hai phiên chạy độc lập nên mỗi phiên chỉ có dòng `summary.csv` của riêng mình. Tệp điểm từng buổi ghi thì không đè nhau — tên cấu hình đã phân biệt (`lstm_...`, `tcn_c64_...`, `ds_tcn_c64_...`).

Ô dưới bung cả hai zip rồi nối hai `summary.csv`, bỏ dòng tiêu đề lặp.


In [4]:
import csv, os, subprocess
DRIVE = "/content/drive/MyDrive/mobivital"

for z in ["tn1_lstm.zip", "tn1_tcn.zip"]:
    assert os.path.exists(DRIVE + "/" + z), "thiếu " + z + " — phiên kia chạy xong chưa?"

# Bung ra hai chỗ riêng rồi mới gộp, để không cái nào đè summary.csv của cái kia.
for z, dich in [("tn1_lstm.zip", "/content/p1"), ("tn1_tcn.zip", "/content/p2")]:
    subprocess.run("rm -rf %s && mkdir -p %s && unzip -qo %s/%s -d %s" % (dich, dich, DRIVE, z, dich), shell=True)

subprocess.run("mkdir -p runs/tn1 runs/tn1_ghij", shell=True)
subprocess.run("cp -r /content/p1/tn1/. runs/tn1/ ; cp -r /content/p2/tn1/. runs/tn1/", shell=True)
subprocess.run("cp -r /content/p1/tn1_ghij/. runs/tn1_ghij/ 2>/dev/null", shell=True)

# Nối hai summary.csv, giữ một dòng tiêu đề, bỏ dòng trùng
rows, header, thay = [], None, set()
for p in ["/content/p1/summary.csv", "/content/p2/summary.csv"]:
    if not os.path.exists(p):
        continue
    r = list(csv.reader(open(p)))
    header = r[0]
    for dong in r[1:]:
        if dong and dong[0] not in thay:
            thay.add(dong[0]); rows.append(dong)

with open("runs/summary.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(header); w.writerows(rows)

print("gộp xong:", len(rows), "dòng trong runs/summary.csv")
for d in sorted(set(r[0] for r in rows)):
    print("   ", d)


gộp xong: 18 dòng trong runs/summary.csv
    ds_tcn_c64_mse_corr0.9_seed0_tong
    ds_tcn_c64_mse_corr0.9_seed0_val_AB
    ds_tcn_c64_mse_corr0.9_seed0_val_CE
    ds_tcn_c64_mse_corr0.9_seed0_val_DF
    ds_tcn_c64_mse_corr0.9_seed0_val_KL
    lstm_mse_corr0.9_seed0
    lstm_mse_corr0.9_seed0_tong
    lstm_mse_corr0.9_seed0_val_AB
    lstm_mse_corr0.9_seed0_val_CE
    lstm_mse_corr0.9_seed0_val_DF
    lstm_mse_corr0.9_seed0_val_KL
    lstm_mse_corr0.9_seed1
    lstm_mse_corr0.9_seed2
    tcn_c64_mse_corr0.9_seed0_tong
    tcn_c64_mse_corr0.9_seed0_val_AB
    tcn_c64_mse_corr0.9_seed0_val_CE
    tcn_c64_mse_corr0.9_seed0_val_DF
    tcn_c64_mse_corr0.9_seed0_val_KL


## 3. So sánh

Hai bảng:

1. **`cv_score`** — điểm trung bình 4 fold của từng cấu hình, kèm chi tiết từng fold và độ lệch chuẩn giữa fold
2. **thắng / hoà / thua** — so **từng** buổi ghi với LSTM, trên đủ 1289 buổi của tám người dev

Bảng 2 cần thiết vì hai cấu hình chênh nhau 0.005 điểm trung bình có thể là tốt hơn đều khắp, hoặc thắng đậm vài buổi mà thua nhẹ phần lớn. Trung bình không phân biệt được. Và nó **không tốn thêm giờ GPU**.

Nếu `cv_score` giữa hai kiến trúc chênh ít hơn `cv_std` thì chưa kết luận được — lúc đó mới cần chạy thêm seed cho hai cấu hình sát nhau.


In [5]:
!python scripts/compare_cv.py --experiment tn1 --baseline lstm



BẢNG 1 — cv_score, thực nghiệm tn1
cấu hình                            tham số   cv_score    cv_std   từng fold
----------------------------------------------------------------------------------------------------
lstm_mse_corr0.9_seed0              1502713   0.760698  0.078486   AB 0.7987  CE 0.7919  DF 0.6265  KL 0.8257
ds_tcn_c64_mse_corr0.9_seed0          56281   0.741376  0.065523   AB 0.7793  CE 0.7857  DF 0.6282  KL 0.7723
tcn_c64_mse_corr0.9_seed0            151513   0.737742  0.085114   AB 0.7847  CE 0.7888  DF 0.5903  KL 0.7872

cv_score = trung bình điểm macro của 4 fold. Điểm macro = trung bình theo
NGƯỜI, không theo buổi ghi — mỗi người có số buổi khác nhau.

BẢNG 2 — thắng / hoà / thua trên TỪNG buổi ghi, mốc là lstm_mse_corr0.9_seed0
cấu hình                            thắng     hoà    thua      tổng   chênh lệch điểm
----------------------------------------------------------------------------------------------------
ds_tcn_c64_mse_corr0.9_seed0          323     620     

## 4. Mốc kiểm chứng trên G H I J

`TN1_LSTM.ipynb` đã chạy 3 seed cho LSTM, được `0.810302 ± 0.015402`. Giờ chạy 3 seed cho **kiến trúc thắng ở mục 3**.

Train đủ tám người `A B C D E F K L` rồi test 537 buổi ghi của `G H I J` — đúng pipeline bài báo dùng, không phải model fold chỉ train 6 người.

**Đây chưa phải số công bố.** Nếu TN2–TN6 đổi cấu hình thì con số này thành cũ và phải chạy lại ở bước cuối. Nó là mốc kiểm chứng của riêng TN1: cho biết hướng đi có đúng không.

Trong hai kiến trúc mới, **DS-TCN-64 thắng** (`cv_score` 0.741376 so với 0.737742 của TCN-64) — nên chạy ô DS-TCN, bỏ qua ô TCN.

Chạy **một** trong hai ô dưới, khoảng 45 phút:


In [ ]:
# chạy ô này nếu TCN thắng ở mục 3
!python scripts/run_final_test.py --experiment tn1_ghij --model tcn --channels 64 --seed 0
!python scripts/run_final_test.py --experiment tn1_ghij --model tcn --channels 64 --seed 1
!python scripts/run_final_test.py --experiment tn1_ghij --model tcn --channels 64 --seed 2


In [6]:
# chạy ô này nếu DS-TCN thắng ở mục 3
!python scripts/run_final_test.py --experiment tn1_ghij --model ds_tcn --channels 64 --seed 0
!python scripts/run_final_test.py --experiment tn1_ghij --model ds_tcn --channels 64 --seed 1
!python scripts/run_final_test.py --experiment tn1_ghij --model ds_tcn --channels 64 --seed 2


thực nghiệm tn1_ghij  -> runs/tn1_ghij/
run_id   ds_tcn_c64_mse_corr0.9_seed0
thiết bị NVIDIA L4

292708 cửa sổ train
56281 tham số

epoch  0  mse 0.04553  pearson 0.4682   0.8 phút
epoch  1  mse 0.02136  pearson 0.5678   1.6 phút
epoch  2  mse 0.01988  pearson 0.5919   2.4 phút
epoch  3  mse 0.01895  pearson 0.6062   3.2 phút
epoch  4  mse 0.01836  pearson 0.6157   4.0 phút
epoch  5  mse 0.01794  pearson 0.6207   4.8 phút
epoch  6  mse 0.01760  pearson 0.6263   5.6 phút
epoch  7  mse 0.01732  pearson 0.6302   6.4 phút
epoch  8  mse 0.01708  pearson 0.6333   7.2 phút
epoch  9  mse 0.01686  pearson 0.6362   8.0 phút
epoch 10  mse 0.01667  pearson 0.6390   8.8 phút
epoch 11  mse 0.01646  pearson 0.6424   9.6 phút
epoch 12  mse 0.01628  pearson 0.6447   10.4 phút
epoch 13  mse 0.01613  pearson 0.6465   11.2 phút
epoch 14  mse 0.01595  pearson 0.6483   12.0 phút
epoch 15  mse 0.01580  pearson 0.6501   12.8 phút
epoch 16  mse 0.01565  pearson 0.6509   13.6 phút
epoch 17  mse 0.01554  pearso

Gộp ba seed thành `mean ± std`:


In [7]:
!python scripts/compare_cv.py --experiment tn1_ghij --final



TEST GHIJ — thực nghiệm tn1_ghij
Train đủ 8 người A B C D E F K L, test 537 buổi ghi của G H I J.

cấu hình                              tham số   seed        mean        std   từng seed
----------------------------------------------------------------------------------------------------
lstm_mse_corr0.9                      1502713      3    0.810302   0.012576   s0 0.8000  s1 0.8029  s2 0.8280
ds_tcn_c64_mse_corr0.9                  56281      3    0.795783   0.012585   s0 0.7811  s1 0.7944  s2 0.8118

std là độ lệch chuẩn giữa các seed — cho biết chênh lệch giữa hai cấu
hình có lớn hơn nhiễu ngẫu nhiên hay không.



## 5. Cất kết quả

Nén cả `runs/tn1/` và `runs/tn1_ghij/` — bản đã gộp đủ ba cấu hình.


In [8]:
!python scripts/save_results.py tn1
!python scripts/save_results.py tn1_ghij


runs/tn1/  ->  runs/tn1.zip   (25.6 MB)
   15 dòng metric trong summary.csv

Bên trong:
        0  2026-09-04 21:01   tn1/
        0  2026-09-04 19:44   tn1/ds_tcn_c64_mse_corr0.9_seed0_val_AB/
        0  2026-09-04 19:44   tn1/ds_tcn_c64_mse_corr0.9_seed0_val_CE/
        0  2026-09-04 19:44   tn1/ds_tcn_c64_mse_corr0.9_seed0_val_DF/
        0  2026-09-04 19:44   tn1/ds_tcn_c64_mse_corr0.9_seed0_val_KL/
        0  2026-09-04 19:44   tn1/lstm_mse_corr0.9_seed0_val_AB/
        0  2026-09-04 19:44   tn1/lstm_mse_corr0.9_seed0_val_CE/
        0  2026-09-04 19:44   tn1/lstm_mse_corr0.9_seed0_val_DF/
        0  2026-09-04 19:44   tn1/lstm_mse_corr0.9_seed0_val_KL/
        0  2026-09-04 19:44   tn1/tcn_c64_mse_corr0.9_seed0_val_AB/
        0  2026-09-04 19:44   tn1/tcn_c64_mse_corr0.9_seed0_val_CE/
        0  2026-09-04 19:44   tn1/tcn_c64_mse_corr0.9_seed0_val_DF/
        0  2026-09-04 19:44   tn1/tcn_c64_mse_corr0.9_seed0_val_KL/
    16949  2026-09-04 19:44   tn1/scores_tcn_c64_mse_corr0.9_

Ghi chú cho thư mục Drive. TN1 chạy ở hai phiên song song nên có cả bản trung gian lẫn bản đã gộp — không có tệp này thì vài tháng sau mở ra không biết cái nào là cái nào.

In [9]:
# Ghi chú cho thư mục Drive, để sau này mở ra biết tệp nào là bản đã gộp.
GHI_CHU = """mobivital/ trên Drive

by_user.tar      12 tệp .npz theo người, dữ liệu đã xử lý
windows.tar.gz   cửa sổ train đã cắt

tn0.zip          TN0 — đối chiếu pipeline đồ án với pipeline MobiVital

tn1.zip          TN1 chọn cấu hình — 3 cấu hình x 4 fold, summary.csv 15 dòng
tn1_ghij.zip     TN1 kiểm chứng GHIJ — lstm x 3 seed, ds_tcn x 3 seed

tn1_lstm.zip     BẢN TRUNG GIAN. Đã gộp vào tn1.zip và tn1_ghij.zip.
tn1_tcn.zip      BẢN TRUNG GIAN. Đã gộp vào tn1.zip.

Hai bản trung gian sinh ra vì TN1 chạy ở hai phiên Colab song song, mỗi phiên
không thấy kết quả của phiên kia nên phải cất riêng. Giữ lại để tra ngược,
không cần dùng nữa — tn1.zip và tn1_ghij.zip đã chứa trọn nội dung của chúng.
"""

open("/content/drive/MyDrive/mobivital/README.txt", "w").write(GHI_CHU)
print(open("/content/drive/MyDrive/mobivital/README.txt").read())
!ls -la /content/drive/MyDrive/mobivital/

mobivital/ trên Drive

by_user.tar      12 tệp .npz theo người, dữ liệu đã xử lý
windows.tar.gz   cửa sổ train đã cắt

tn0.zip          TN0 — đối chiếu pipeline đồ án với pipeline MobiVital

tn1.zip          TN1 chọn cấu hình — 3 cấu hình x 4 fold, summary.csv 15 dòng
tn1_ghij.zip     TN1 kiểm chứng GHIJ — lstm x 3 seed, ds_tcn x 3 seed

tn1_lstm.zip     BẢN TRUNG GIAN. Đã gộp vào tn1.zip và tn1_ghij.zip.
tn1_tcn.zip      BẢN TRUNG GIAN. Đã gộp vào tn1.zip.

Hai bản trung gian sinh ra vì TN1 chạy ở hai phiên Colab song song, mỗi phiên
không thấy kết quả của phiên kia nên phải cất riêng. Giữ lại để tra ngược,
không cần dùng nữa — tn1.zip và tn1_ghij.zip đã chứa trọn nội dung của chúng.

total 2771617
-rw------- 1 root root 2640629760 Sep  3 15:32 by_user.tar
-rw------- 1 root root        819 Sep  4 21:17 README.txt
-rw------- 1 root root    5643034 Sep  4 00:57 tn0.zip
-rw------- 1 root root   17485723 Sep  4 21:01 tn1_ghij.zip
-rw------- 1 root root   39092589 Sep  4 16:57 tn1_lstm.zip